---
last_verified: 2026-08-21
tool_version: n/a
---

# Terraform State Isolation — comparison notebook

This notebook compares three common patterns for isolating Terraform state across environments: workspaces, named environments, and remote-state isolation with separate backends.

## Purpose

Terraform state isolation prevents one environment from accidentally destroying resources in another. Three patterns dominate practice: Terraform workspaces (single config, multiple state files), named environments (separate directory trees per environment), and remote-state isolation (dedicated backend per environment with locking). Each pattern has different tradeoffs in complexity, blast radius, and team collaboration.

## When to use

- **Workspaces** — single team, similar environments, low configuration overhead. Works well when environments share the same resources and you want one codebase.
- **Named environments** — distinct teams per environment, or environments with materially different resource shapes. Directory separation reduces accidental cross-environment drift.
- **Remote-state isolation** — strict separation required, multi-team ownership of environments, or compliance rules that mandate independent state storage and locking.

## Prerequisites

- Terraform CLI available in the execution environment
- Bash shell (examples use shell heredocs)
- For remote-state examples: an S3-compatible object store and a DynamoDB-compatible table for locking (examples show configuration only; no real cloud resources are created)

## 1 — Workspaces

Workspaces are built into Terraform. `terraform workspace new <name>` creates a new isolated state file under `.terraform/workspaces/`. The same configuration file is used for every workspace; only the state differs.

In [ ]:
%%bash

# Workspace demo: create a temp dir, init, create a workspace, and show state isolation
WORKDIR=$(mktemp -d)
cat > "$WORKDIR/main.tf" <<"EOF"
terraform {
  required_version = ">= 1.0"
}

resource "null_resource" "env_marker" {
  triggers = {
    workspace = terraform.workspace
  }
}
EOF

cd "$WORKDIR"
terraform init -input=false > /dev/null 2>&1

# Create a dev workspace
terraform workspace new dev 2>/dev/null || true
terraform apply -auto-approve -input=false > /dev/null 2>&1

# Show state for dev workspace
echo "=== dev workspace state ==="
terraform show -no-color 2>/dev/null | grep -A1 "env_marker" || true

# Switch to prod workspace and apply again
terraform workspace new prod 2>/dev/null || true
terraform apply -auto-approve -input=false > /dev/null 2>&1

echo "=== prod workspace state ==="
terraform show -no-color 2>/dev/null | grep -A1 "env_marker" || true

# List workspaces
echo "=== all workspaces ==="
terraform workspace list

cd /work
rm -rf "$WORKDIR"

### Verify

- `terraform workspace list` shows both `dev` and `prod`.
- `terraform show` inside each workspace displays only that workspace's state.
- Deleting the workspace removes its state file entirely.

## 2 — Named environments (separate directories)

Instead of one config and multiple workspaces, you maintain a directory per environment. Each directory has its own `main.tf`, `terraform.tfvars`, and backend block. This pattern treats environments as first-class directory artifacts.

In [ ]:
%%bash

# Named-environment demo: scaffold dev/ and prod/ directories
WORKDIR=$(mktemp -d)
mkdir -p "$WORKDIR/dev" "$WORKDIR/prod"

cat > "$WORKDIR/dev/main.tf" <<"EOF"
terraform {
  required_version = ">= 1.0"
}

resource "null_resource" "dev_marker" {
  triggers = {
    env = "dev"
  }
}
EOF

cat > "$WORKDIR/prod/main.tf" <<"EOF"
terraform {
  required_version = ">= 1.0"
}

resource "null_resource" "prod_marker" {
  triggers = {
    env = "prod"
  }
}
EOF

echo "=== directory layout ==="
find "$WORKDIR" -type f | sort

# Init and apply in dev
cd "$WORKDIR/dev"
terraform init -input=false > /dev/null 2>&1
terraform apply -auto-approve -input=false > /dev/null 2>&1
echo "=== dev state ==="
terraform show -no-color 2>/dev/null | grep -A1 "dev_marker" || true

# Init and apply in prod
cd "$WORKDIR/prod"
terraform init -input=false > /dev/null 2>&1
terraform apply -auto-approve -input=false > /dev/null 2>&1
echo "=== prod state ==="
terraform show -no-color 2>/dev/null | grep -A1 "prod_marker" || true

cd /work
rm -rf "$WORKDIR"

### Verify

- Each directory has its own `terraform.tfstate`.
- Resources in `dev/` do not appear in `prod/` state.
- Changing a resource in one directory never affects the other.

## 3 — Remote-state isolation

Remote-state isolation uses a dedicated backend per environment, typically with an S3 bucket for state and a DynamoDB table for locking. Each environment writes to a unique state key, ensuring that even if the same configuration directory is reused, state never crosses environment boundaries.

In [ ]:
# Example S3 backend configuration for dev and prod
# (not executed — requires real AWS credentials and resources)
#
# File: envs/dev/backend.tf
# terraform {
#   backend "s3" {
#     bucket         = "company-terraform-state"
#     key            = "dev/terraform.tfstate"
#     region         = "us-east-1"
#     dynamodb_table = "terraform-locks"
#     encrypt        = true
#   }
# }
#
# File: envs/prod/backend.tf
# terraform {
#   backend "s3" {
#     bucket         = "company-terraform-state"
#     key            = "prod/terraform.tfstate"
#     region         = "us-east-1"
#     dynamodb_table = "terraform-locks"
#     encrypt        = true
#   }
# }
#
# Key points:
# - Same bucket, different key = isolated state files
# - dynamodb_table provides distributed locking across all environments
# - encrypt = true enables server-side encryption on the state object

### Verify

- `terraform init` points to the correct backend.
- State keys are distinct: `dev/terraform.tfstate` vs `prod/terraform.tfstate`.
- DynamoDB lock entries appear per operation; concurrent runs are serialized.

## Comparison

| Dimension | Workspaces | Named environments | Remote-state isolation |
|-----------|-----------|-------------------|----------------------|
| State isolation | Same config, different workspace state files | Separate local state files per directory | Separate remote state keys per environment |
| Configuration duplication | None — single codebase | Higher — each directory repeats backend and provider blocks | Low — backend block differs, rest can be shared via modules |
| Blast radius | Medium — wrong workspace switches are easy | Low — wrong directory must be cd'd into | Low — distinct backend keys and separate credentials |
| Team collaboration | Hard — everyone shares one config history | Medium — teams can own directories independently | Best — environments can live in separate repos or branches |
| Locking | No built-in cross-workspace locking | Local only unless combined with a remote backend | Built-in via DynamoDB or equivalent |
| Best for | Single-team greenfield apps | Distinct environment shapes or legacy splits | Regulated multi-team platforms with strict separation requirements |

## Common errors

- **Workspace name collision:** `terraform workspace select` fails silently if you assume the wrong workspace. Always list workspaces before destructive operations.
- **State leakage across directories:** named environments with a shared local backend can drift if a developer accidentally runs `terraform apply` from the wrong directory. Use shell aliases or wrapper scripts that enforce the target directory.
- **Backend migration without locking:** moving from local to remote state mid-project requires `terraform init -backend-config` and a state push. Without locking enabled during migration, concurrent runs can corrupt state.
- **Hard-coded backend credentials:** remote-state backends should pull credentials from environment variables or a secrets manager, never from committed `.tf` files or `terraform.tfvars`.

## How this connects to what's next

State isolation is the foundation for confident CI/CD pipelines. Once environments are properly isolated, the next step is automating plan and apply with approval gates, or introducing policy-as-code tools like OPA to enforce backend and workspace rules across the organization.